# Stage 3 - Evidence-guided correction and the controlled ablation
Eight systems answer the *same* questions (paired design):

| id | system | what it adds |
|---|---|---|
| A | LLM only | - |
| B | LLM + RAG | retrieval |
| C | LLM + self-consistency | SelfCheck-NLI flags -> flagged claims removed |
| D | LLM + verification | CoVe (no retrieval) flags -> removed |
| E | LLM + RAG + verification | NLI + LLM verifier vs. retrieved evidence -> removed |
| F | E + correction | RARR-style rewrite from evidence, **verify-then-accept** with NLI, citations |
| G | F with the hybrid detector | adds SelfCheck signal to the detector |
| F-CoVe | E + CoVe-style correction | alternative correction strategy |

If RAG abstains, E/F/G verify the parametric answer against retrieved evidence (`pipeline.rag_fallback_to_parametric`). Cost of a system = the stages it depends on (shared stages replay their cost), so latency/tokens/calls are what each system would spend alone.

**Evaluation**: an independent stronger judge grades CORRECT / INCORRECT / NOT_ATTEMPTED against *gold* references (so abstaining cannot inflate scores) and labels each claim SUPPORTED / CONTRADICTED / UNVERIFIABLE (FActScore-style precision). Judge-free proxies are always reported too.

In [ ]:
import os, sys
from pathlib import Path
import matplotlib; matplotlib.use("Agg")          # figures are written to results/ and shown with IPython.display.Image
from IPython.display import Image, display, Markdown
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", "{:.3f}".format)

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# ---- choose how to run -----------------------------------------------------------------------------------------
PROFILE = None          # None = configs/default.yaml | "configs/quick.yaml" = small sanity run (recommended first)
MOCK = os.environ.get("HALLUGUARD_MOCK") == "1"      # offline smoke test with FAKE models - numbers are meaningless
from halluguard.config import load_config
cfg = load_config(PROFILE, mock=MOCK)
print("mock:", MOCK, "| generator:", cfg.llm.model, "| judge:", cfg.llm.judge_model, "| results ->", cfg.project.results_dir)

def show(name, stage):
    p = Path(cfg.project.results_dir) / stage / name
    display(Image(filename=str(p))) if p.exists() else print("(missing)", p)


In [ ]:
from halluguard.budget import print_budget
print_budget(cfg)
# Judge options on a free key (70B judge = 100K tokens/day):  USE_JUDGE=False  |  cfg.llm.judge_model = "llama-3.1-8b-instant" (self-judging, flagged in meta.json)
USE_JUDGE = cfg.stage3.use_llm_judge
SYSTEMS = None          # e.g. ["A","B","E","F"] for a cheaper subset

In [ ]:
from halluguard.experiments.common import load_resources
from halluguard.experiments import stage3
res = load_resources(cfg)
r3 = stage3.run(cfg, res, systems=SYSTEMS, use_judge=USE_JUDGE)

## 1. Headline table

In [ ]:
s = r3["summary_overall"]
cols = ["system","n","correct","incorrect","not_attempted","correct_given_attempted","f_score","claim_support_rate","claim_contradiction_rate","factscore","free_correct","free_incorrect","cost_calls","cost_total_tokens","cost_latency"]
s[[c for c in cols if c in s]]

In [ ]:
show("grades.png", "stage3")

## 2. Which component helps? Paired contrasts with bootstrap CIs and McNemar tests
Each row changes exactly one component. Differences whose CI contains 0 are not evidence of an effect at this sample size.

In [ ]:
c = r3["contrasts"]
c[c.scope == "all"][["contrast","n","delta_correct","correct_lo","correct_hi","delta_incorrect","incorrect_lo","incorrect_hi","mcnemar_p"]]

In [ ]:
show("marginal_contributions.png", "stage3")

## 3. Factuality vs. cost (the trade-off asked for in the proposal)

In [ ]:
show("pareto.png", "stage3"); show("cost.png", "stage3")

## 4. Per-dataset results, claim-level factuality, flagging statistics

In [ ]:
d = r3["summary"]; d[[c for c in ["dataset","system","correct","incorrect","not_attempted","f_score","claim_support_rate","free_correct","abstention_rate"] if c in d]]

In [ ]:
show("claim_rates.png", "stage3"); show("flag_stats.png", "stage3"); show("threshold_retention.png", "stage3")

### TruthfulQA split by whether the KB holds the question's Wikipedia source

In [ ]:
pd.read_csv(Path(cfg.project.results_dir) / "stage3" / "summary_truthfulqa_by_wiki_coverage.csv")[["tqa_has_wiki","system","n","abstention_rate","correct","incorrect","free_correct"]]

## 5. Qualitative examples (initial vs. corrected, with citations) - including failures


In [ ]:
print((Path(cfg.project.results_dir) / "stage3" / "examples.md").read_text()[:6000])

## 6. Try your own question
Same pipeline as the Streamlit demo (`streamlit run app.py`).

In [ ]:
question = "What happens to you if you eat watermelon seeds?"
from halluguard.pipeline import FactualityPipeline, QuestionState
from halluguard.factory import load_calibration
pipe = FactualityPipeline(cfg, res.llm, res.retriever, res.nli, load_calibration(cfg))
outs = pipe.run_all(QuestionState("demo", question), ["A", "B", "F"])
for sid, o in outs.items():
    print(f"--- {sid} ({o.initial_source}) | flagged {o.n_flagged}, removed {o.n_removed}, rewritten {o.n_rewritten} | {o.cost['calls']:.0f} calls, {o.cost['total_tokens']:.0f} tokens")
    print(o.final_answer_cited)

## Reporting checklist
* State sample sizes (`meta.json`), the judge model and whether judge == generator (`judge_equals_generator`), and that the KB is an oracle-in-pool setting.
* Do not report a run whose `meta.json` says `"mock": true`.
* Interpret contrasts through their CIs; with n~80 questions only large effects are detectable - raise `stage3.n_per_dataset` if the budget allows.